In [12]:
pip uninstall dquant

^C
Note: you may need to restart the kernel to use updated packages.


In [13]:
pip install dquant==1.1.4b0

  Attempting uninstall: dquant
    Found existing installation: dquant 1.1.3.1
    Uninstalling dquant-1.1.3.1:
      Successfully uninstalled dquant-1.1.3.1
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Installation of dependencies

In [3]:
pip install dquant arch yfinance

  Using cached yfinance-1.3.0-py2.py3-none-any.whl.metadata (6.1 kB)
  Using cached multitasking-0.0.13-py3-none-any.whl.metadata (16 kB)
  Using cached pytz-2026.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached frozendict-2.4.7-py3-none-any.whl.metadata (23 kB)
  Using cached peewee-4.0.5-py3-none-any.whl.metadata (8.6 kB)
  Using cached curl_cffi-0.15.0-cp310-abi3-win_amd64.whl.metadata (18 kB)
  Using cached websockets-16.0-cp312-cp312-win_amd64.whl.metadata (7.0 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
Using cached yfinance-1.3.0-py2.py3-none-any.whl (133 kB)
Using cached curl_cffi-0.15.0-cp310-abi3-win_amd64.whl (1.7 MB)
Using cached frozendict-2.4.7-py3-none-any.whl (16 kB)
Using cached multitasking-0.0.13-py3-none-any.whl (16 kB)
Using cached peewee-4.0.5-py3-none-any.whl (144 kB)
Using cached pytz-2026.2-py2.py3-none-any.whl (510 kB)
Using cached websockets-16.0-cp312-cp312-win_amd64.whl


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Example with DQuant

In [10]:
"""
Честное сравнение DQuant vs GARCH(1,1)
- Обучаем модели на данных до 2023-01-01
- Тестируем на данных 2023-01-01 – 2024-12-31
- Горизонт прогноза = 1 день
- Единая метрика качества – Parkinson volatility
- Статистическая значимость – тест Диболда-Мариано
"""

import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime
from sklearn.metrics import mean_absolute_error
from dquant.models import VolClustXGB
from arch import arch_model
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 1. ПАРАМЕТРЫ
# ============================================================
TICKER = 'SPY'                  # Тикер (можно BTC-USD, EURUSD=X, GC=F)
START_DATE = '2014-01-01'       # Начало всей выборки
SPLIT_DATE = '2023-01-01'       # Дата разделения train/test
END_DATE = '2024-12-31'         # Конец тестовой выборки

# DQuant
INPUT_BARS = 70                 # Сколько свечей подавать на вход
HORIZON = 20                     # Горизонт прогноза в днях
TREES_COUNT = 2               # Максимальное число деревьев для early stopping
FEATURES = [                    # Список признаков (из документации)
    'returns'
]

# ============================================================
# 2. ЗАГРУЗКА ДАННЫХ И ПОДГОТОВКА
# ============================================================
print(f"Загрузка {TICKER} с {START_DATE} по {END_DATE}...")
raw = yf.download(TICKER, start=START_DATE, end=END_DATE, auto_adjust=True)

# Приводим к формату DQuant: open, high, low, close, volume
df = pd.DataFrame({
    'open':   raw[('Open', TICKER)].values,
    'high':   raw[('High', TICKER)].values,
    'low':    raw[('Low', TICKER)].values,
    'close':  raw[('Close', TICKER)].values,
    'volume': raw[('Volume', TICKER)].values
}, index=raw.index)
print(df)
# Дневная доходность и Parkinson volatility (прокси реализованной волатильности)
df['returns'] = df['close'].pct_change()
df['parkinson_vol'] = np.sqrt(
    (1 / (4 * np.log(2))) * (np.log(df['high'] / df['low']))**2
)
df = df.dropna().copy()

print(f"Всего свечей: {len(df)}")
print(f"Период: {df.index[0].date()} – {df.index[-1].date()}")

# ============================================================
# 3. РАЗДЕЛЕНИЕ НА TRAIN И TEST (СТАТИЧНЫЙ СПЛИТ)
# ============================================================
train_mask = df.index < SPLIT_DATE
test_mask = df.index >= SPLIT_DATE

df_train = df[train_mask].copy()
df_test = df[test_mask].copy()

print(f"\nTrain: {len(df_train)} свечей (до {SPLIT_DATE})")
print(f"Test:  {len(df_test)} свечей (с {SPLIT_DATE})")
print(df_train)
# ============================================================
# 4. ОБУЧЕНИЕ DQUANT (ОДИН РАЗ НА TRAIN)
# ============================================================
def parkinson_func(df):
  df = df.copy()
  tr_values = []

  for i in range(1, len(df['high'])):
    #print(1)
    #print(i-1)
    #print(len(df['high']))
    tr_values.append(np.sqrt(
        (1 / (4 * np.log(2))) * (np.log(df['high'].iloc[i] / df['low'].iloc[i]))**2
    ))
    #print(2)

  return np.array(tr_values)


print("\n=== Обучение DQuant ===")
model_dq = VolClustXGB({}, early_stopping=True)
model_dq.fit(
    df_train,
    feature_list=FEATURES,
    input_bars=INPUT_BARS,
    horizon=HORIZON,
    trees_count=TREES_COUNT,
    show_results=False,
    target_func=parkinson_func
)
print("DQuant обучен.")

# ============================================================
# 5. ПРОГНОЗЫ DQUANT НА ТЕСТОВОМ ПЕРИОДЕ
# ============================================================
print("\n=== Прогноз DQuant на тестовом периоде ===")
dquant_preds = []
dquant_dates = []

# Идём по каждому дню тестового периода, для которого возможен прогноз
for i in range(INPUT_BARS, len(df_test)):
    # Текущая дата тестового дня
    test_date = df_test.index[i]

    # Берём все доступные данные до test_date (train + предыдущие дни test)
    available_data = df.loc[:test_date].iloc[-INPUT_BARS:].copy()
    if len(available_data) < INPUT_BARS:
        continue   # недостаточно истории

    try:
        forecast_vals = model_dq.forecast(available_data, show=False)
        # forecast_vals – массив длины HORIZON
        pred_vol = forecast_vals[-1] if HORIZON > 1 else forecast_vals[0]
    except Exception as e:
        print(f"Ошибка прогноза DQuant на {test_date.date()}: {e}")
        continue
# Фактическое значение Parkinson vol на test_date
    actual_vol = df_test.loc[test_date, 'parkinson_vol']

    dquant_dates.append(test_date)
    dquant_preds.append({
        'date': test_date,
        'actual': actual_vol,
        'predicted': pred_vol
    })

df_dquant = pd.DataFrame(dquant_preds).set_index('date')
print(f"DQuant прогнозов: {len(df_dquant)}")

# ============================================================
# 6. ПРОГНОЗЫ GARCH (РАСШИРЯЮЩЕЕСЯ ОКНО ДЛЯ ЧЕСТНОСТИ)
# ============================================================
print("\n=== Прогноз GARCH ===")
garch_preds = []
garch_dates = []

# Нам нужна история доходностей до каждого тестового дня
returns_full = df['returns'].dropna() * 100   # масштабирование для GARCH

for test_date in df_test.index:
    # Все доходности до test_date (включая train и прошлые дни test)
    hist_returns = returns_full[:test_date].iloc[:-1]  # не включаем сам test_date
    if len(hist_returns) < 500:
        continue   # слишком мало данных для обучения

    try:
        model_garch = arch_model(hist_returns, vol='GARCH', p=1, q=1, dist='normal')
        fitted = model_garch.fit(disp='off')
        forec = fitted.forecast(horizon=1, reindex=False)
        cond_var = forec.variance.values[-1, 0]
        pred_vol = np.sqrt(cond_var) / 100.0   # обратное масштабирование
    except Exception as e:
        print(f"Ошибка GARCH на {test_date.date()}: {e}")
        continue

    actual_vol = df.loc[test_date, 'parkinson_vol']
    garch_preds.append({
        'date': test_date,
        'actual': actual_vol,
        'predicted': pred_vol
    })

df_garch = pd.DataFrame(garch_preds).set_index('date')
print(f"GARCH прогнозов: {len(df_garch)}")
print(df_garch)
# ============================================================
# 7. СРАВНЕНИЕ НА ОБЩИХ ДАТАХ
# ============================================================
common_dates = df_dquant.index.intersection(df_garch.index)
print(f"\nОбщих дат для сравнения: {len(common_dates)}")

dquant_aligned = df_dquant.loc[common_dates]
garch_aligned = df_garch.loc[common_dates]

# Метрики
mae_dq = mean_absolute_error(dquant_aligned['actual'], dquant_aligned['predicted'])
mae_garch = mean_absolute_error(garch_aligned['actual'], garch_aligned['predicted'])

def qlike(y_true, y_pred):
    sigma2_true = y_true**2
    sigma2_pred = np.maximum(y_pred**2, 1e-10)
    return np.mean(np.log(sigma2_pred) + sigma2_true / sigma2_pred)

qlike_dq = qlike(dquant_aligned['actual'], dquant_aligned['predicted'])
qlike_garch = qlike(garch_aligned['actual'], garch_aligned['predicted'])

# Тест Диболда-Мариано (на квадратах ошибок)
err_dq = (dquant_aligned['actual'] - dquant_aligned['predicted'])**2
err_garch = (garch_aligned['actual'] - garch_aligned['predicted'])**2
d = err_dq - err_garch
n = len(d)
# Простая реализация без учёта автокорреляции (h=1)
dm_stat = np.mean(d) / np.std(d, ddof=1) * np.sqrt(n)
p_value = 2 * (1 - stats.norm.cdf(abs(dm_stat)))

# ============================================================
# 8. РЕЗУЛЬТАТЫ
# ============================================================
print("\n========== РЕЗУЛЬТАТЫ ==========")
print(f"MAE   DQuant: {mae_dq:.6f}   GARCH: {mae_garch:.6f}")
print(f"QLIKE DQuant: {qlike_dq:.6f}   GARCH: {qlike_garch:.6f}")
print(f"DM-тест: статистика = {dm_stat:.4f}, p-value = {p_value:.4f}")
if p_value < 0.05:
    print("=> Различие статистически значимо (на уровне 5%).")
else:
    print("=> Различие не является статистически значимым.")

# Сохранение прогнозов для графиков
#dquant_aligned.to_csv('dquant_test_predictions.csv')
#garch_aligned.to_csv('garch_test_predictions.csv')
#print("\nФайлы dquant_test_predictions.csv и garch_test_predictions.csv сохранены.")

[*********************100%***********************]  1 of 1 completed

Загрузка SPY с 2014-01-01 по 2024-12-31...
                  open        high         low       close     volume
Date                                                                 
2014-01-02  149.441311  149.514424  148.222907  148.580307  119636900
2014-01-03  148.832046  149.132594  148.344692  148.555878   81390600
2014-01-06  149.043249  149.100102  147.897947  148.125381  108028200
2014-01-07  148.718357  149.286942  148.604639  149.035141   86144200
2014-01-08  149.010762  149.319428  148.555893  149.067627   96582300
...                ...         ...         ...         ...        ...
2024-12-23  582.440932  586.787848  579.257077  586.186584   57635800
2024-12-24  587.537023  592.741554  586.955433  592.702087   33160100
2024-12-26  590.927860  593.865231  589.528182  592.741577   41219100
2024-12-27  588.995868  589.232487  582.312845  586.502075   64969300
2024-12-30  579.483905  583.278831  576.053624  579.809143   56578800

[2767 rows x 5 columns]
Всего свечей: 2766
Пер

1
0
70
2
1
1
70
2
1
2
70
2
1
3
70
2
1
4
70
2
1
5
70
2
1
6
70
2
1
7
70
2
1
8
70
2
1
9
70
2
1
10
70
2
1
11
70
2
1
12
70
2
1
13
70
2
1
14
70
2
1
15
70
2
1
16
70
2
1
17
70
2
1
18
70
2
1
19
70
2
1
20
70
2
1
21
70
2
1
22
70
2
1
23
70
2
1
24
70
2
1
25
70
2
1
26
70
2
1
27
70
2
1
28
70
2
1
29
70
2
1
30
70
2
1
31
70
2
1
32
70
2
1
33
70
2
1
34
70
2
1
35
70
2
1
36
70
2
1
37
70
2
1
38
70
2
1
39
70
2
1
40
70
2
1
41
70
2
1
42
70
2
1
43
70
2
1
44
70
2
1
45
70
2
1
46
70
2
1
47
70
2
1
48
70
2
1
49
70
2
1
50
70
2
1
51
70
2
1
52
70
2
1
53
70
2
1
54
70
2
1
55
70
2
1
56
70
2
1
57
70
2
1
58
70
2
1
59
70
2
1
60
70
2
1
61
70
2
1
62
70
2
1
63
70
2
1
64
70
2
1
65
70
2
1
66
70
2
1
67
70
2
1
68
70
2
Подготовка данных: |░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░|   0.00%  Осталось 75.96 секунд1
0
70
2
1
1
70
2
1
2
70
2
1
3
70
2
1
4
70
2
1
5
70
2
1
6
70
2
1
7
70
2
1
8
70
2
1
9
70
2
1
10
70
2
1
11
70
2
1
12
70
2
1
13
70
2
1
14
70
2
1
15
70
2
1
16
70
2
1
17
70
2
1
18
70
2
1
19
70
2
1
20
70
2
1
21
70
2
1
22
70


KeyboardInterrupt: 